In [ ]:
%%capture
!pip install elasticsearch

In [ ]:
from elasticsearch import Elasticsearch
import json
import re
from elasticsearch.helpers import bulk
from elasticsearch import Elasticsearch, helpers


In [ ]:
# Connect
import os
API_KEY = os.environ.get("ELASTIC_API_KEY", "")
es = Elasticsearch(ENDPOINT,api_key=API_KEY)

print("Connected successfully!") if es.ping() else print("Connection failed.")

Connected successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SAVEFILE_DIR = "/content/drive/MyDrive/Mystery Search Engine/"

In [ ]:
# Dataset File
def load_from_jsonl(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            yield json.loads(line)
dataset_jsonl = load_from_jsonl(f'{SAVEFILE_DIR}flat_wiki_1.jsonl')

### Fresh Start (Delete Old Dataset & Replace with new dataset)

In [ ]:
INDEX_NAME = "pages"

In [ ]:
# 1. DELETE OLD INDEX (Start Fresh)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"Deleted existing index: {INDEX_NAME}")

Deleted existing index: pages


In [ ]:
# 2. CREATE NEW INDEX WITH MAPPING
mapping = {
    "mappings": {
        "properties": {
            "page_title": {
                "type": "text",
                "fields": {
                    "keyword": {"type": "keyword"} # This creates page_title.keyword
                }
            },
            "section_title": {"type": "text"},
            "breadcrumb":    {"type": "text"},
            "content":       {"type": "text"},
            "infobox":       {"type": "text"},
            "category":      {"type": "keyword"},
            "location":      {"type": "keyword"},
            "is_lead":       {"type": "boolean"},
            "last_update":   {"type": "text"}
        }
    }
}
es.indices.create(index=INDEX_NAME, body=mapping)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'pages'})

In [ ]:
# 3. BULK UPLOAD FROM JSONL
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            yield {
                "_index": INDEX_NAME,
                "_source": json.loads(line)
            }

filename = f'{SAVEFILE_DIR}flat_wiki_1.jsonl'
print("Starting bulk upload...")
helpers.bulk(es, load_jsonl(filename))
print("Upload Complete!")

Starting bulk upload...
Upload Complete!


### Search Function

In [ ]:
def search_wikipedia(
    query_text,           # str: search query text
    size=10,              # int: Number of retrieved document
    search_fields=None,   # list: what field to search, default=all
    filters=None,         # dict: Filters (Exact Match), default=None
    unique_pages=False,    # bool: force retrieve unique pages
    output_fields=None,   # list: output what field, default=All
    output_raw=False      # bool: output raw response
  ):
    if not search_fields:
        search_fields = ["page_title^5", "section_title^3", "infobox^2", "content"]

    # 1. Base Query
    body = {
        "size": size,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query_text,
                        "fields": search_fields,
                        "type": "best_fields"
                    }
                },
                "filter": []
            }
        }
    }

    # 2. Add Filters
    if filters:
        for field, value in filters.items():
            if isinstance(value, list):
                body["query"]["bool"]["filter"].append({"terms": {field: value}})
            else:
                body["query"]["bool"]["filter"].append({"term": {field: value}})

    # 3. UNIQUE PAGES (Field Collapsing)
    if unique_pages:
        # We collapse on 'page_title.keyword' to ensure uniqueness
        body["collapse"] = {
            "field": "page_title.keyword"
        }

    # 4. Output Fields
    if output_fields:
        body["_source"] = output_fields

    res = es.search(index=INDEX_NAME, body=body)

    if output_raw:
      return res

    return [hit["_source"] for hit in res["hits"]["hits"]]

In [ ]:
# example filters
# filters={"category": "murder", "location": "United States"}

In [ ]:
a = search_wikipedia("jack ripper",size=20,unique_pages=True)

In [ ]:
a[0]

{'page_title': 'Jack the Ripper',
 'section_title': 'Summary',
 'breadcrumb': ['Jack the Ripper'],
 'content': 'Jack the Ripper was an unidentified serial killer active in and around the impoverished Whitechapel district of London, England, in 1888. In both criminal case files and the contemporaneous journalistic accounts, the killer was also called the Whitechapel Murderer and Leather Apron.\nAttacks ascribed to Jack the Ripper typically involved women working as prostitutes who lived in the slums of the East End of London. Their throats were cut prior to abdominal mutilations. The removal of internal organs from at least three of the victims led to speculation that the killer had some anatomical or surgical knowledge. Rumours that the murders were connected intensified in September and October 1888, and numerous letters were received by media outlets and Scotland Yard from people purporting to be the murderer.\nThe name "Jack the Ripper" originated in the "Dear Boss letter" written b

In [ ]:
len(a)

10

### Tidak dipakai

In [ ]:
error guard
# What is inside response?

user_query = "West Ridge Road"  # You can change this term to test

# Run a match query against 'title' and 'summary' fields
response = es.search(
    index=INDEX_NAME,
    query={
        "multi_match": {
            "query": user_query,
            "fields": ["title", "summary"]
        }
    }
)

# Print the results
print(f"\n--- Search results for '{user_query}' ---")
hits = response['hits']['hits']

if not hits:
    print("No matching documents found.")
else:
    for hit in hits:
        score = hit["_score"]  # Relevancy score
        source = hit["_source"]
        print(f"Score: {score:.2f}")
        print(f"Title: {source['title']}")
        print(f"Summary: {source['summary']}")
        print(f"Location: {source['SE_Location']}")
        print("-" * 30)


--- Search results for 'West Ridge Road' ---
Score: 4.17
Title: 1978 Holiday Inn fire
Summary: The 1978 Holiday Inn fire broke out at a Holiday Inn hotel located at 1525 West Ridge Road in the town of Greece, New York, United States, on November 26, 1978. The fire was considered notable enough by the National Fire Protection Association (NFPA) and the Center for Fire Research to document the fire in their 1979 publications. In the end, ten people were killed and 34 injured; seven of the fatalities were Canadian nationals. In 2008, the NFPA listed the 1978 Holiday Inn fire as one of only three dozen or so hotel fires which killed ten or more people in the U.S. between 1934 and 2006.
Location: United States
------------------------------


In [ ]:
hit

{'_index': 'pages',
 '_id': '3',
 '_score': 0.7140372,
 '_source': {'title': '2003 Columbus, Ohio arson',
  'summary': 'On 13 April 2003, a fire was set at a house in the University District of Columbus, Ohio, killing five university students (two from Ohio State University and three from Ohio University). Despite an arrest early on, the murders have remained unsolved.',
  'SE_Location': 'United States'}}

In [ ]:
# Define your search query
user_query = "fire"  # You can change this term to test
location_filter = "United States"

# Run a match query against 'title' and 'summary' fields
response = es.search(
    index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {
                    "multi_match": {
                        "query": user_query,
                        "fields": ["title", "summary"]
                    }
                }
            ],
            "filter": [
                {
                    "term": {
                        "SE_Location.keyword": location_filter
                    }
                }
            ]
        }
    }
)

# Print the results
print(f"\n--- Search results for '{user_query}' ---")
hits = response['hits']['hits']

if not hits:
    print("No matching documents found.")
else:
    for hit in hits:
        score = hit["_score"]  # Relevancy score
        source = hit["_source"]
        print(f"Score: {score:.2f}")
        print(f"Title: {source['title']}")
        print(f"Summary: {source['summary']}")
        print(f"Location: {source['SE_Location']}")
        print("-" * 30)


--- Search results for 'fire' ---
Score: 0.99
Title: 1978 Holiday Inn fire
Summary: The 1978 Holiday Inn fire broke out at a Holiday Inn hotel located at 1525 West Ridge Road in the town of Greece, New York, United States, on November 26, 1978. The fire was considered notable enough by the National Fire Protection Association (NFPA) and the Center for Fire Research to document the fire in their 1979 publications. In the end, ten people were killed and 34 injured; seven of the fatalities were Canadian nationals. In 2008, the NFPA listed the 1978 Holiday Inn fire as one of only three dozen or so hotel fires which killed ten or more people in the U.S. between 1934 and 2006.
Location: United States
------------------------------
Score: 0.71
Title: 2003 Columbus, Ohio arson
Summary: On 13 April 2003, a fire was set at a house in the University District of Columbus, Ohio, killing five university students (two from Ohio State University and three from Ohio University). Despite an arrest earl